# 🦆 Microduck 강화학습 시작하기

**강화학습이 처음이어도 괜찮아요.** 위에서부터 셀을 하나씩 실행(▶ 또는 `Shift+Enter`)하면,
가상 오리 로봇이 **넘어지기만 하다가 → 스스로 걷는 법을 배우는 과정**을 직접 보게 됩니다.

| 단계 | 하는 일 | 걸리는 시간 |
|---|---|---|
| 0 | 강화학습 3분 개념 | 읽기 |
| 1 | GPU 켜기 + 설치 | 약 10분 (처음 한 번) |
| 2 | **맛보기 학습** → 영상으로 "아기 오리" 보기 | 약 5~10분 |
| 3 | **본 학습** → 걷는 오리 | 1시간~ (GPU에 따라) |
| 4 | 그래프로 학습 과정 이해하기 | 읽기 |
| 5 | 결과를 파일로 받아 Mac에서 키보드로 조종 | 5분 |

> 사용하는 코드: Pollen Robotics 공식 [`microduck_rl`](https://github.com/pollen-robotics/microduck_rl) (mjlab + PPO).
> 이 노트북은 그걸 쉽게 돌리기 위한 안내서입니다.

## 0. 강화학습 3분 개념

**아기가 걸음마를 배우는 것과 같아요.** 아무도 "무릎을 30도 굽혀"라고 가르쳐 주지 않습니다.
그냥 이것저것 해보다가 **잘 되면 칭찬(+점수), 넘어지면 꾸중(−점수)** 을 받고, 점점 칭찬받는 쪽으로 움직임이 바뀝니다.

```
        ┌──────────── 관측 (지금 몸 상태) ────────────┐
        │                                              ▼
 ┌──────────────┐     행동 (관절 14개 목표 각도)   ┌──────────────┐
 │  환경         │ ◀───────────────────────────── │  정책 (뇌)     │
 │ MuJoCo 물리   │                                │  작은 신경망    │
 │ 시뮬레이터     │ ──── 보상 (잘했나? 점수) ────▶ │               │
 └──────────────┘                                └──────────────┘
        이걸 1초에 50번, 오리 수천 마리가 동시에 반복 → 점수가 오르는 쪽으로 신경망을 조금씩 수정
```

| 용어 | 뜻 | Microduck에서는 |
|---|---|---|
| **정책 (policy)** | 행동을 정하는 "뇌" | 작은 신경망. 학습의 결과물이 바로 이것 (`.pt`, `.onnx` 파일) |
| **환경 (environment)** | 연습하는 세상 | MuJoCo 물리 시뮬레이터 속 가상 오리 |
| **관측 (observation)** | 뇌가 보는 정보 | 몸 기울기, 회전 속도, 관절 각도, "이 속도로 걸어" 명령 등 |
| **행동 (action)** | 뇌가 내리는 결정 | 관절 14개의 목표 각도, 1초에 50번 |
| **보상 (reward)** | 잘했는지 점수 | 아래 표 참고 |
| **에피소드 (episode)** | 한 판 | 넘어지거나 20초(=1000스텝)가 지나면 처음부터 다시 |
| **병렬 환경 (num envs)** | 동시에 연습하는 오리 수 | GPU 덕분에 4096마리가 한꺼번에 연습 |
| **반복 (iteration)** | 한 번의 공부 | 오리들이 24스텝씩 경험을 모으면 → 신경망을 한 번 수정 |
| **PPO** | 신경망을 고치는 방법(알고리즘) | 점수가 좋았던 행동은 더, 나빴던 행동은 덜 하도록 "조금씩" 고침 |

### 이 오리가 받는 점수 (보상 함수)

걷기 태스크(`Mjlab-Velocity-Flat-MicroDuck`)의 주요 항목입니다. **강화학습에서 가장 중요한 건 "무엇에 점수를 주느냐"** 예요.

| 보상 항목 | 가중치 | 의미 |
|---|---|---|
| `track_linear_velocity` | +2.0 | 명령받은 속도로 앞/옆으로 가면 + |
| `track_angular_velocity` | +2.0 | 명령받은 속도로 돌면 + |
| `upright` | +2.0 | 몸을 똑바로 세우고 있으면 + |
| `air_time` | +3.0 | 발을 적당히 들었다 놓으면(걸음) + — 이게 없으면 발을 질질 끌며 미끄러지는 꼼수를 배움 |
| `pose` | +1.0 | 기본 자세에서 너무 벗어나지 않으면 + |
| `head_pose_tracking` | +2.0 | 머리를 명령한 방향으로 두면 + |
| `foot_slip` | −0.1 | 발이 바닥에서 미끄러지면 − |
| `self_collisions` | −1.0 | 자기 몸끼리 부딪히면 − |
| `body_ang_vel`, `angular_momentum` | −0.05, −0.02 | 몸이 마구 흔들리면 − |

넘어지면(`fell_over`) 그 판은 바로 끝나서 더 이상 점수를 못 받아요. 그래서 **오래 버티는 것 자체가 이득**이 됩니다.

## 1. 준비

### 1-1. GPU 켜기
메뉴 **런타임 → 런타임 유형 변경 → 하드웨어 가속기: GPU** (무료는 T4, 구독 중이면 L4/A100 선택 가능) → 저장.

아래 셀을 실행해서 GPU 이름이 나오면 OK.

In [ ]:
import subprocess
try:
    print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True))
    print('✅ GPU 준비 완료')
except Exception:
    print('❌ GPU가 없어요. 런타임 → 런타임 유형 변경 → GPU 를 선택한 뒤 다시 실행하세요.')

### 1-2. 설치 (처음 한 번, 약 10분)

하는 일:
1. **Google Drive 연결** — 학습 결과를 Drive(`내 드라이브/microduck/`)에 저장해서, Colab이 끊겨도 날아가지 않게 합니다. 권한 창이 뜨면 허용하세요.
2. **uv** (파이썬 패키지 설치 도구) 설치
3. 공식 `microduck_rl` 코드 받기
4. 필요한 라이브러리(PyTorch, MuJoCo, mjlab 등, 수 GB) 설치

> Colab은 세션이 끝나면 설치한 게 사라져요. 다음에 다시 열면 **이 셀부터 다시 실행**하면 됩니다. (학습 결과는 Drive에 남아 있어요.)

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/microduck'
os.makedirs(f'{DRIVE}/logs', exist_ok=True)
os.makedirs(f'{DRIVE}/videos', exist_ok=True)

!curl -LsSf https://astral.sh/uv/install.sh | sh > /dev/null
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
os.environ['UV_HTTP_TIMEOUT'] = '600'
os.environ['MUJOCO_GL'] = 'egl'          # 화면 없이 영상 렌더링

%cd /content
!test -d microduck_rl || git clone -q https://github.com/pollen-robotics/microduck_rl
%cd /content/microduck_rl
!git pull -q; git log -1 --format='코드 버전: %h %s (%cd)' --date=short
!rm -rf logs && ln -s {DRIVE}/logs logs     # 학습 기록은 Drive에 저장

print('📦 라이브러리 설치 중... (5~10분)')
!uv sync 2>&1 | tail -2
print('✅ 설치 완료')

### 1-3. 도우미 함수 불러오기

학습 로그를 보기 쉽게 요약하고, 영상을 찍고, 체크포인트를 찾는 함수들이에요. **내용은 몰라도 됩니다.** 실행만 하세요.

In [ ]:
#@title 도우미 함수 (실행만 하면 됨)
import os, re, glob, time, shutil, signal, subprocess, collections
from IPython.display import Video, display

TASK = 'Mjlab-Velocity-Flat-MicroDuck'
ANSI = re.compile(r'\x1b\[[0-9;]*m')

def _num(line):
    try: return float(line.split()[-1])
    except Exception: return None

def train(run_name, num_envs, iterations, every=10, resume_from=None):
    """학습 실행. 전체 로그는 Drive에 저장하고, 화면에는 핵심만 요약해서 보여줌."""
    cmd = ['uv', 'run', 'train', TASK,
           '--env.scene.num-envs', str(num_envs),
           '--agent.max-iterations', str(iterations),
           '--agent.save-interval', '50',          # 50번마다 체크포인트 저장 (끊겨도 안전)
           '--agent.logger', 'tensorboard',
           '--agent.run-name', run_name]
    if resume_from:
        cmd += ['--agent.resume', 'True',
                '--agent.load-run', os.path.basename(os.path.dirname(resume_from)),
                '--agent.load-checkpoint', os.path.basename(resume_from)]
    log_path = f'{DRIVE}/train_{run_name}.log'
    print(f'🏋️ 학습 시작: 오리 {num_envs}마리 × {iterations}번 반복  (전체 로그: {log_path})\n')
    print('       반복 | 평균 점수 |          버틴 시간 | 속도 오차 | 남은 시간')
    print('-' * 70)
    tail = collections.deque(maxlen=40)
    s = {}
    with open(log_path, 'a') as logf, subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as p:
        for raw in p.stdout:
            logf.write(raw); line = ANSI.sub('', raw).strip(); tail.append(line)
            m = re.search(r'Learning iteration (\d+)/(\d+)', line)
            if m: s = {'it': int(m[1]), 'total': int(m[2])}
            elif line.startswith('Mean reward:'): s['rew'] = _num(line)
            elif line.startswith('Mean episode length:'): s['len'] = _num(line)
            elif line.startswith('Metrics/twist/error_vel_xy:'): s['err'] = _num(line)
            elif line.startswith('ETA:') and 'it' in s:
                if s['it'] % every == 0 or s['it'] == s['total'] - 1:
                    f = lambda k, fmt: format(s[k], fmt) if s.get(k) is not None else '-'
                    print(f"{s['it']:>5}/{s['total']:<5} | {f('rew', '9.2f')} | {f('len', '6.0f')} / 1000 스텝 | {f('err', '9.3f')} | {line.split()[-1]}", flush=True)
    if p.returncode != 0:
        print('\n❌ 학습이 오류로 멈췄어요. 마지막 로그:\n' + '\n'.join(tail))
    else:
        print('\n✅ 학습 완료! 마지막 체크포인트:', latest_checkpoint(run_name))

def checkpoints(run_name):
    """해당 run의 체크포인트를 반복 횟수 순으로."""
    files = glob.glob(f'logs/rsl_rl/*/*_{run_name}/model_*.pt')
    return sorted(files, key=lambda f: int(re.search(r'model_(\d+)\.pt', f)[1]))

def latest_checkpoint(run_name):
    c = checkpoints(run_name)
    return c[-1] if c else None

def record_video(ckpt, seconds=6, name=None):
    """체크포인트로 오리를 움직여 영상을 찍고 화면에 보여줌."""
    if not ckpt: print('체크포인트가 없어요. 먼저 학습하세요.'); return
    vdir = os.path.join(os.path.dirname(ckpt), 'videos', 'play')
    shutil.rmtree(vdir, ignore_errors=True)
    it = re.search(r'model_(\d+)\.pt', ckpt)[1]
    print(f'🎥 {it}번 반복한 오리를 촬영 중... (1~2분)')
    cmd = ['uv', 'run', 'play', TASK, '--checkpoint-file', ckpt, '--num-envs', '1',
           '--video', 'True', '--video-length', str(seconds * 50), '--viewer', 'viser']
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, start_new_session=True)
    out, start, size = None, time.time(), -1
    while time.time() - start < 600 and p.poll() is None:
        time.sleep(3)
        mp4 = glob.glob(f'{vdir}/*.mp4')
        if mp4:
            new = os.path.getsize(mp4[0])
            if new == size and new > 0: out = mp4[0]; break   # 파일 쓰기가 끝남
            size = new
    try: os.killpg(p.pid, signal.SIGTERM)    # 영상만 필요하니 뷰어는 종료
    except Exception: pass
    if not out:
        print('❌ 영상을 못 만들었어요. 로그:'); print(ANSI.sub('', p.stdout.read() if p.stdout else '')[-3000:]); return
    dst = f'{DRIVE}/videos/{name or "iter" + it}.mp4'
    shutil.copy(out, dst)
    print(f'저장됨: {dst}')
    display(Video(dst, embed=True, width=480))

print('✅ 도우미 함수 준비 완료')

## 2. 맛보기 학습: "아기 오리" 만들기 (5~10분)

일단 **짧게** 학습해서 전체 흐름을 익혀 봅시다. 오리 1024마리가 100번 공부합니다.

출력되는 표 읽는 법:

| 열 | 뜻 | 좋아지면 |
|---|---|---|
| **평균 점수** | 한 판 동안 받은 보상 합계 | ⬆ 올라감 |
| **버틴 시간** | 한 판을 평균 몇 스텝 버텼나 (최대 1000 = 20초) | ⬆ 올라감 = 덜 넘어짐 |
| **속도 오차** | 명령한 속도와 실제 속도 차이 (m/s) | ⬇ 내려감 |

> 처음엔 거의 바로 넘어져서 버틴 시간이 수십 스텝 정도예요. 100번만 해도 숫자가 변하는 게 보일 거예요.

In [ ]:
train('quick', num_envs=1024, iterations=100, every=10)

### 2-1. 아기 오리 영상 보기

방금 학습한 오리를 6초간 촬영합니다. 아마 **비틀거리거나 넘어질** 거예요. 아직 100번밖에 안 배웠으니까요.
(오리는 무작위로 주어지는 "이 속도로 걸어" 명령을 따르려고 애씁니다.)

In [ ]:
record_video(latest_checkpoint('quick'), name='01_baby_duck')

## 3. 본 학습: 걷는 오리 만들기

이제 제대로 학습합니다. 공식 설정은 **오리 4096마리**이고, 좋은 GPU에서 1~2시간이면 쓸 만한 걸음이 나온다고 해요.

**설정 고르기**

| GPU | `NUM_ENVS` | 팁 |
|---|---|---|
| T4 (무료) | 2048 (메모리 부족 오류 나면 1024) | 느려요. 몇 시간 걸릴 수 있으니 끊기면 3-2로 이어하기 |
| L4 / A100 (구독) | 4096 | 공식 설정 그대로 |

`ITERATIONS`는 일단 **2000**으로 두고, 표의 **남은 시간**을 보고 판단하세요.
50번마다 Drive에 저장되니까 **중간에 멈춰도(■ 버튼) 그때까지의 결과는 남아요.**

> 💡 무료 Colab은 브라우저 탭을 닫거나 오래 자리를 비우면 끊길 수 있어요. 탭을 열어 두세요.

In [ ]:
NUM_ENVS = 4096
ITERATIONS = 2000

train('main', num_envs=NUM_ENVS, iterations=ITERATIONS, every=25)

### 3-1. 걷는 오리 영상 보기

In [ ]:
record_video(latest_checkpoint('main'), name='02_walking_duck')

### 3-2. (끊겼을 때) 이어서 학습하기

Colab이 끊겼다면: **1-2 설치 → 1-3 도우미 함수**를 다시 실행한 다음 아래 셀을 실행하세요.
Drive에 저장된 마지막 체크포인트부터 `ITERATIONS`번 **더** 학습합니다.

In [ ]:
last = latest_checkpoint('main')
print('이어서 학습할 체크포인트:', last)
train('main', num_envs=NUM_ENVS, iterations=1000, every=25, resume_from=last)

### 3-3. 성장 과정 비교하기

저장된 체크포인트 중 몇 개를 골라 촬영합니다. 반복 횟수가 늘수록 걸음이 어떻게 바뀌는지 보세요.
(체크포인트당 1~2분 걸려요.)

In [ ]:
ckpts = checkpoints('main')
print(f'체크포인트 {len(ckpts)}개:', [re.search(r"model_(\d+)", c)[1] for c in ckpts])
picks = [ckpts[0], ckpts[len(ckpts) // 3], ckpts[2 * len(ckpts) // 3], ckpts[-1]] if len(ckpts) >= 4 else ckpts
for c in picks:
    record_video(c, seconds=5)

## 4. 그래프로 학습 과정 이해하기 (TensorBoard)

아래 셀을 실행하면 그래프 화면이 열려요. 왼쪽 검색창에 이름을 넣어 찾으세요.

| 그래프 | 보는 법 |
|---|---|
| `Train/mean_reward` | 평균 점수. **우상향**이면 학습 중 |
| `Train/mean_episode_length` | 버틴 시간. **1000에 가까워지면** 거의 안 넘어짐 |
| `Episode_Termination/fell_over` | 넘어져서 끝난 비율. **내려가야** 좋음 |
| `Metrics/twist/error_vel_xy` | 속도 오차. **내려가야** 좋음 |
| `Episode_Reward/...` | 보상 항목별 점수. 어떤 칭찬을 많이 받는지 보여줌 |
| `Policy/mean_std` | 행동의 무작위성(탐험). 학습이 진행될수록 **줄어듦** = 자신감이 생김 |

> 점수가 오르다 평평해지면 그 설정으로는 거의 다 배운 거예요. 점수는 높은데 영상이 이상하면? 보상 함수의 "꼼수"를 찾은 것 — 강화학습에서 흔한 일이에요.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/rsl_rl

## 5. 결과물 받기 → Mac에서 키보드로 조종

학습한 정책을 실제 로봇과 같은 형식(**ONNX**)으로 내보내요. 반드시 공식 `export.py`를 쓰세요. 관측값 정규화가 파일 안에 함께 들어가야 로봇·시뮬레이터에서 제대로 동작합니다.

In [ ]:
ckpt = latest_checkpoint('main')
print('내보낼 체크포인트:', ckpt)
!uv run scripts/export.py {TASK} --checkpoint-file {ckpt} 2>&1 | tail -1
it = re.search(r'model_(\d+)', ckpt)[1]
dst = f'{DRIVE}/walking_iter{it}.onnx'
shutil.copy('output.onnx', dst)
print('✅ 저장됨:', dst, '→ Mac의 Google Drive 폴더나 drive.google.com에서 받으세요')

Mac에서 (`microduck_rl` 폴더, 설치법은 [워크스페이스 README](https://github.com/fredisbusy/microduck)):

```sh
cd ~/Projects/microduck/microduck_rl
uv run scripts/infer_policy.py --walking ~/Downloads/walking_iter2000.onnx
```
MuJoCo 창이 뜨면 **터미널을 클릭한 상태에서** 키보드로 조종: **↑/↓** 앞뒤 속도, **←/→** 회전. 내가 키운 오리예요!

## 6. 다음에 해볼 것

1. **다른 묘기 배우기** — 맨 위 도우미 함수 셀의 `TASK`를 바꾸고 다시 실행:

   | 태스크 | 배우는 것 |
   |---|---|
   | `Mjlab-StandUp-Flat-MicroDuck` | 넘어진 상태에서 일어나기 |
   | `Mjlab-SitStand-Flat-MicroDuck` | 앉았다 일어서기 |
   | `Mjlab-BallKick-Flat-MicroDuck` | 공 차기 |
   | `Mjlab-Roulade-Flat-MicroDuck` | 앞구르기 |
   | `Mjlab-Velocity-Flat-MicroDuck-Rollers` | 롤러스케이트 타기 |
   | `Mjlab-VelStand-Rough-Backlash-MicroDuck` | 실제 로봇이 쓰는 걷기 (울퉁불퉁 바닥 + 기어 유격) |

   전체 목록: `!uv run list-envs`

2. **보상 바꿔 보기** — `src/mjlab_microduck/tasks/microduck_velocity_env_cfg.py`에서 `cfg.rewards["air_time"].weight = 3.0` 같은 줄을 찾아 숫자를 바꾸고 다시 학습해 보세요. 예: `air_time`을 0으로 하면 오리가 발을 끌기 시작하는지?
3. **Mac에서 빠르게 실험** — [Microduck Lab](https://github.com/jonathanhawkins/microduck-lab)은 GPU 없이 Mac에서 몇 분 만에 학습하고 브라우저로 볼 수 있어요.

## 문제 해결

| 증상 | 해결 |
|---|---|
| `CUDA out of memory` | `NUM_ENVS`를 절반으로 (4096 → 2048 → 1024) |
| "GPU 사용 한도에 도달" | 무료 한도 초과. 몇 시간~하루 뒤 다시, 또는 구독 컴퓨팅 단위 사용 |
| 세션이 끊김 | 1-2, 1-3 다시 실행 → 3-2 이어서 학습 |
| 영상이 안 만들어짐 | 1-2 설치 셀의 `MUJOCO_GL=egl` 설정이 실행됐는지 확인 후 1-3부터 다시 |